In [ ]:
from pathlib import Path

from satchange.automated_hpo import run_automated_parameter_search
from satchange.run import resolve_parameters
from satchange.utils import load_config

cwd = Path.cwd().resolve()
project_root = cwd.parents[1] if cwd.parent.name == "notebooks" else cwd
config_path = project_root / "configs/siam_mit_b3_diff_full_resolution.yaml"
config = load_config(config_path)
dataset_root = project_root / config["dataset"]["prepared_directory"]
HPO_ROOT = project_root / "artifacts/hyperparameter_search/heavy/heavy_full"

GPU_IDS = [0, 1, 2, 3]  # Укажи только выделенные и свободные GPU.
THRESHOLDS = [round(value / 100, 2) for value in range(5, 96, 5)]

print("Config:", config_path)
print("Dataset:", dataset_root)
print("Image size:", config["data"]["image_size"], "— исходное разрешение")
print("Checkpointing:", config["model"]["parameters"]["use_checkpointing"])

In [ ]:
HPO_STAGES = [
    ("01_batch_lr", {
        "batch_size": [1, 2, 4, 8],
        "learning_rate": [3e-5, 1e-4, 3e-4],
    }),
    ("02_lr", {
        "learning_rate": [1e-4, 3e-4, 6e-4],
    }),
    ("03_loss", {
        "positive_weight": [5, 10],
        "loss_weights": [(0.3, 0.7), (0.5, 0.5)],
    }),
    ("04_wd_scheduler", {
        "weight_decay": [1e-3, 1e-2],
        "scheduler": ["none", "reduce_on_plateau"],
    }),
]


In [3]:
summary = run_automated_parameter_search(
    stages=HPO_STAGES,
    base_parameters=resolve_parameters(config),
    model_name=config["model"]["name"],
    dataset_root=dataset_root,
    thresholds=THRESHOLDS,
    number_of_epochs=config["training"]["epochs"],
    patience=12,
    min_epochs=20,
    min_delta=0.001,
    output_root=HPO_ROOT,
    search_name="heavy_full_hpo_40ep_es_v2",
    gpu_ids=GPU_IDS,
    f1_gap=0.04,
    iou_gap=0.05,
    seeds=[42, 123, 777],
)

best = summary["selected_by_mean_validation_f1"]
print("Параметры:", best["parameters"])
print("Validation F1:", best["mean_f1"], "+/-", best["std_f1"])
print("Validation IoU:", best["mean_iou"], "+/-", best["std_iou"])
print("Результаты:", summary["search_directory"])



[Trial 000 | GPU 3] Epoch 38/40
Train loss:       0.189209
Validation loss:  0.227575
Best threshold:   0.70
Precision:        0.806734
Recall:           0.813943
F1:               0.810323
IoU:              0.681128

[Trial 000 | GPU 3] Epoch 39/40
Train loss:       0.190812
Validation loss:  0.225772
Best threshold:   0.65
Precision:        0.818375
Recall:           0.810994
F1:               0.814668
IoU:              0.687291

[Trial 000 | GPU 3] Epoch 40/40
Train loss:       0.198266
Validation loss:  0.235155
Best threshold:   0.65
Precision:        0.790228
Recall:           0.805012
F1:               0.797552
IoU:              0.663273

Trial 0 finished on GPU 3
Parameters: seed=42 | lr=3e-05 | wd=0.01 | pw=5 | BCE/Dice=0.5/0.5 | batch=1 | image_size=None | attention_dropout=0.0 | decoder_channels=128 | drop_path_rate=0.1 | dropout=0.0 | encoder_depths=[3, 4, 18, 3] | in_channels=3 | norm_groups=8 | out_channels=1 | pretrained_backbone=False | refinement_channels=[64, 32] | u

In [1]:
from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile
import importlib.util
import json
import sys

import torch
from torch._subclasses.fake_tensor import FakeTensorMode
from IPython.display import FileLink, display


# Запускай из корня проекта или его notebooks/.
cwd = Path.cwd().resolve()
relative_root = Path("artifacts/hyperparameter_search/heavy")

project_root = next(
    (
        path
        for path in (cwd, *cwd.parents)
        if (path / relative_root).is_dir()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        f"Не найден {relative_root}. Открой notebook внутри проекта."
    )

heavy_root = project_root / relative_root
archive_path = (
    cwd
    / f"heavy_hpo_diagnostic_{datetime.now():%Y%m%d_%H%M%S_%f}.zip"
)

report = {
    "started_at_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(project_root),
    "heavy_root": str(heavy_root),
    "python": sys.version,
    "executable": sys.executable,
    "torch_version": str(torch.__version__),
    "files": [],
    "checkpoints": [],
    "source_files": [],
    "errors": [],
}


def json_default(value):
    # Значения тензоров в отчёт не включаем.
    if isinstance(value, torch.Tensor):
        return {
            "tensor_shape": list(value.shape),
            "dtype": str(value.dtype),
        }
    return str(value)


def file_stamp(path):
    stat = path.stat()
    return stat.st_ino, stat.st_size, stat.st_mtime_ns


def checkpoint_info(path):
    before = file_stamp(path)

    # Только собственные checkpoints: weights_only=False использует pickle.
    with FakeTensorMode():
        state = torch.load(
            path,
            map_location="cpu",
            weights_only=False,
        )

    if not isinstance(state, dict):
        raise TypeError("Checkpoint не является словарём.")

    fields = (
        "epoch",
        "model_name",
        "run_name",
        "config",
        "precision",
        "world_size",
        "best_metrics",
        "current_metrics",
        "early_stopping",
        "stop_reason",
        "resumed_from",
        "thresholds",
        "history",
    )

    info = {key: state[key] for key in fields if key in state}

    info["path"] = str(path.resolve())
    info["relative_path"] = path.relative_to(heavy_root).as_posix()
    info["metadata_readable"] = True
    info["changed_during_read"] = before != file_stamp(path)

    # Отмечаем наличие состояний, но сами состояния в архив не кладём.
    info["state_fields_present"] = {
        key: key in state and state[key] is not None
        for key in (
            "model_state_dict",
            "optimizer_state_dict",
            "scheduler_state_dict",
            "rng_states",
            "best_model_state_dict",
            "scaler_state_dict",
        )
    }

    return json.loads(json.dumps(info, default=json_default))


# Если настройки уже есть в текущем notebook, добавляем их.
# Если нет — ячейка всё равно работает.
report["notebook_context"] = {
    name: globals()[name]
    for name in (
        "HPO_ROOT",
        "SEARCH_NAME",
        "HPO_STAGES",
        "GPU_IDS",
        "THRESHOLDS",
        "config_path",
        "config",
        "RESUME_BASE_PARAMETERS",
        "BASE_PARAMETERS",
        "BEST_CHECKPOINTS",
        "CHECKPOINTS",
        "TARGET_EPOCHS",
    )
    if name in globals()
}

report["loaded_satchange_modules"] = {
    name: str(module.__file__)
    for name, module in list(sys.modules.items())
    if name.startswith("satchange") and getattr(module, "__file__", None)
}


with ZipFile(archive_path, "x", compression=ZIP_DEFLATED) as archive:

    # 1. Все поиски внутри heavy: файлы, checkpoints и JSON-результаты.
    for path in sorted(heavy_root.rglob("*")):
        if not path.is_file():
            continue

        try:
            before = file_stamp(path)

            record = {
                "path": str(path.resolve()),
                "relative_path": path.relative_to(heavy_root).as_posix(),
                "size_bytes": before[1],
                "mtime_ns": before[2],
            }
            report["files"].append(record)

            if path.suffix.lower() in {".pt", ".pth"}:
                info = checkpoint_info(path)
                report["checkpoints"].append(info)

                print(
                    f"epoch={str(info.get('epoch', '?')):>4}"
                    f" | {record['relative_path']}",
                    flush=True,
                )

            elif path.suffix.lower() in {".json", ".csv", ".yaml", ".yml"}:
                archive.writestr(
                    path.relative_to(project_root).as_posix(),
                    path.read_bytes(),
                )

            record["changed_during_read"] = before != file_stamp(path)

        except Exception as error:
            report["errors"].append({
                "path": str(path),
                "error": f"{type(error).__name__}: {error}",
            })
            print(f"Не прочитан: {path.name} — {error}", flush=True)

    # 2. Исходники: основная копия, изолированная и найденная в окружении.
    source_roots = {
        "project": project_root / "src/satchange",
        "isolated": project_root / ".hpo_parallel_runtime/src/satchange",
    }

    try:
        spec = importlib.util.find_spec("satchange")
        if spec is not None and spec.origin:
            source_roots["environment"] = Path(spec.origin).resolve().parent
    except (ImportError, ValueError) as error:
        report["errors"].append({
            "path": "satchange import",
            "error": str(error),
        })

    report["source_roots"] = {
        name: str(path)
        for name, path in source_roots.items()
    }

    for label, source_root in source_roots.items():
        for path in sorted(source_root.rglob("*.py")):
            try:
                data = path.read_bytes()
                archive_name = (
                    f"sources/{label}/satchange/"
                    f"{path.relative_to(source_root).as_posix()}"
                )

                archive.writestr(archive_name, data)
                report["source_files"].append({
                    "path": str(path),
                    "archive_path": archive_name,
                    "sha256": sha256(data).hexdigest(),
                })

            except Exception as error:
                report["errors"].append({
                    "path": str(path),
                    "error": str(error),
                })

    # 3. YAML-конфигурации.
    for path in sorted((project_root / "configs").rglob("*")):
        if path.is_file() and path.suffix.lower() in {".yaml", ".yml"}:
            try:
                archive.writestr(
                    path.relative_to(project_root).as_posix(),
                    path.read_bytes(),
                )
            except Exception as error:
                report["errors"].append({
                    "path": str(path),
                    "error": str(error),
                })

    # 4. Итоговый отчёт с абсолютными путями и метаданными checkpoints.
    report["finished_at_utc"] = datetime.now(timezone.utc).isoformat()
    report["note"] = (
        "Metadata-only diagnostic, NOT a checkpoint backup or validation "
        "of tensor contents. Running training may change files during/after "
        "inspection. No training was started."
    )

    archive.writestr(
        "report.json",
        json.dumps(
            report,
            indent=2,
            ensure_ascii=False,
            default=json_default,
        ),
    )


print(f"\nCheckpoints прочитано: {len(report['checkpoints'])}")
print(f"Ошибок чтения: {len(report['errors'])}")
print("Архив:", archive_path)

display(FileLink(archive_path.name))

epoch=  30 | heavy_full/heavy_full_hpo_40ep_es_v2/01_batch_lr_4623f6b45262/siam_mit_b3_diff/trial_000/best.pt
epoch=  40 | heavy_full/heavy_full_hpo_40ep_es_v2/01_batch_lr_4623f6b45262/siam_mit_b3_diff/trial_000/checkpoints/siam_mit_b3_diff_epoch_0040.pt
epoch=  40 | heavy_full/heavy_full_hpo_40ep_es_v2/01_batch_lr_4623f6b45262/siam_mit_b3_diff/trial_000/last.pt
epoch=  38 | heavy_full/heavy_full_hpo_40ep_es_v2/01_batch_lr_4623f6b45262/siam_mit_b3_diff/trial_001/best.pt
epoch=  40 | heavy_full/heavy_full_hpo_40ep_es_v2/01_batch_lr_4623f6b45262/siam_mit_b3_diff/trial_001/checkpoints/siam_mit_b3_diff_epoch_0040.pt
epoch=  40 | heavy_full/heavy_full_hpo_40ep_es_v2/01_batch_lr_4623f6b45262/siam_mit_b3_diff/trial_001/last.pt
epoch=  38 | heavy_full/heavy_full_hpo_40ep_es_v2/01_batch_lr_4623f6b45262/siam_mit_b3_diff/trial_002/best.pt
epoch=  40 | heavy_full/heavy_full_hpo_40ep_es_v2/01_batch_lr_4623f6b45262/siam_mit_b3_diff/trial_002/checkpoints/siam_mit_b3_diff_epoch_0040.pt
epoch=  40 | he

/home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/notebooks/heavy_hpo_diagnostic_20260929_160840_626839.zip

In [ ]:
summary = run_final_benchmark(
    variant="heavy_resized",
    checkpoints="artifacts/final_models/medium_full",

    dataset_root="data/LEVIR-CD",
    device="auto",
    evaluation_batch_size=4,
    num_workers=0,
    precision="fp32",

    trusted_checkpoints=True,   # Только собственные доверенные checkpoints.
    overwrite=True,            # Обновить существующую папку, сохранив backup.
)

print("Результаты сохранены:", summary["output_dir"])